# Gold Layer — Build fact_sales
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Grain** | One row per order line item (`order_items`) |
| **Source** | `silver.order_items`, `silver.orders`, `silver.products`, `silver.payments`, `silver.address`, `gold.dim_date` |
| **Target** | `gbmart.gold.fact_sales` |

### Shape — matches the original mockup column-for-column
```
fact_sales
├── fact_sales_sk        (surrogate PK — the only generated key in this table)
├── Payment_ID           (natural key, from silver.payments)
├── Customer_ID          (natural key)
├── Product_ID           (natural key)
├── Order_ID             (natural key)
├── Address_ID           (natural key)
├── Time_ID              (natural key — dim_date.date_key)
├── Quantity_purchased   (measure)
├── Actual_price         (measure)
├── Discounted_price     (measure)
└── Sales_amount         (measure = Quantity_purchased x Discounted_price)
```

> **Note on this design choice:** every FK here is a natural/business key
> (`Customer_ID`, `Product_ID`, etc.), not a dimension surrogate key. This
> means `fact_sales` doesn't snapshot *which version* of a customer/product
> was current when the order happened — it just stores the natural ID.
> `dim_customer`/`dim_product` still exist as proper SCD2 dimensions with
> their own surrogate keys and history; this fact table just isn't wired
> to them through that mechanism. That's a deliberate simplification for
> this training build, not an oversight.

## Step 1 — Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

CATALOG = "gbmart"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

## Step 2 — Read the Grain + Order Header Info
`order_items` is the grain. Join `orders` to get `Customer_ID` and
`order_date` (needed to look up `Time_ID`).

In [0]:
order_items_df = spark.table("gbmart.silver.order_items") \
    .select(
        col("order_item_id"),
        col("order_id").alias("Order_ID"),
        col("product_id").alias("Product_ID"),
        col("quantity").alias("Quantity_purchased")
    )

orders_df = spark.table("gbmart.silver.orders") \
    .select(col("order_id").alias("Order_ID"), col("customer_id").alias("Customer_ID"), "order_date")

base_df = order_items_df.join(orders_df, "Order_ID")
print(f"Base rows: {base_df.count():,}")

## Step 3 — Look Up `Actual_price` and `Discounted_price`
Join `silver.products` on the natural `Product_ID`, filtered to
`is_current = true` so each product resolves to exactly one price row.

In [0]:
products_current = spark.table("gbmart.silver.products").filter("is_current = true") \
    .select(
        col("product_id").alias("Product_ID"),
        col("actual_price_inr").alias("Actual_price"),
        col("discounted_price_inr").alias("Discounted_price")
    )

enriched_df = base_df.join(products_current, "Product_ID")
print(f"Rows after products join: {enriched_df.count():,}")

## Step 4 — Look Up `Time_ID`
Join on `order_date` to resolve `dim_date.date_key`, used here as `Time_ID`.

In [0]:
dim_date_df = spark.table("gbmart.gold.dim_date").select("date", col("date_key").alias("Time_ID"))

# Drop any leftover Time_ID/date columns from a previous run of this cell —
# makes this cell safe to re-run without restarting from Step 3
enriched_df = enriched_df.drop("Time_ID", "date")

enriched_df = enriched_df.join(dim_date_df, enriched_df.order_date == dim_date_df.date, "left")
print(f"Rows after dim_date join: {enriched_df.count():,}")
print(f"Rows with no matching Time_ID: {enriched_df.filter(col('Time_ID').isNull()).count():,}")

## Step 5 — Look Up `Address_ID`

`silver.address` links to `Customer_ID`, not to a specific order — a
customer can have more than one address (Billing, Shipping, or both). We
pick one per customer: prefer an address whose `address_type` includes
`"Shipping"`, otherwise take the first one available.

In [0]:
address_window = Window.partitionBy("customer_id").orderBy(
    when(col("address_type").contains("Shipping"), 0).otherwise(1)
)

address_primary = spark.table("gbmart.silver.address") \
    .withColumn("_rank", row_number().over(address_window)) \
    .filter("_rank = 1") \
    .select(col("customer_id").alias("Customer_ID"), col("address_id").alias("Address_ID"))

enriched_df = enriched_df.join(address_primary, "Customer_ID", "left")
print(f"Rows after address join: {enriched_df.count():,}")
print(f"Rows with no matching Address_ID: {enriched_df.filter(col('Address_ID').isNull()).count():,}")

## Step 6 — Look Up `Payment_ID`
Payments are 1:1 with orders — join `silver.payments` on `Order_ID` to get
the natural `Payment_ID` directly.

In [0]:
payments_df = spark.table("gbmart.silver.payments") \
    .select(col("order_id").alias("Order_ID"), col("payment_id").alias("Payment_ID"))

enriched_df = enriched_df.join(payments_df, "Order_ID", "left")
print(f"Rows after payments join: {enriched_df.count():,}")
print(f"Rows with no matching Payment_ID: {enriched_df.filter(col('Payment_ID').isNull()).count():,}")

## Step 7 — Compute `Sales_amount` and Final Select
`fact_sales_sk` is the one and only surrogate key in this table — generated
from `order_item_id`, since that's the fact's natural grain key.

In [0]:
fact_sales_df = enriched_df \
    .withColumn("fact_sales_sk", sha2(col("order_item_id"), 256)) \
    .withColumn("Sales_amount", col("Quantity_purchased") * col("Discounted_price")) \
    .select(
        "fact_sales_sk", "Payment_ID", "Customer_ID", "Product_ID", "Order_ID", "Address_ID",
        "Time_ID", "Quantity_purchased", "Actual_price", "Discounted_price", "Sales_amount"
    )

print(f"fact_sales rows: {fact_sales_df.count():,}")
fact_sales_df.display()

## Step 8 — Write to Gold

In [0]:
fact_sales_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.fact_sales")
print(f"Written: {spark.table('gbmart.gold.fact_sales').count():,} rows")

In [0]:
%sql
CREATE OR REPLACE VIEW gbmart.gold.vw_monthly_category_sales AS
SELECT
    d.year,
    d.month,
    p.category,
    p.sub_category,
    SUM(f.Quantity_purchased)                          AS total_quantity_sold,
    SUM(f.Sales_amount)                                AS total_revenue,
    COUNT(DISTINCT f.Order_ID)                         AS total_orders,
    ROUND(AVG(f.Actual_price - f.Discounted_price), 2) AS avg_discount_given
FROM gbmart.gold.fact_sales f
JOIN gbmart.gold.dim_product p ON f.Product_ID = p.product_id AND p.is_current = true
JOIN gbmart.gold.dim_date d    ON f.Time_ID = d.date_key
GROUP BY d.year, d.month, p.category, p.sub_category;

In [0]:
%sql
select * from gbmart.gold.vw_monthly_category_sales;

## Reset (if needed)

In [0]:
# spark.sql("DROP TABLE IF EXISTS gbmart.gold.fact_sales")
# print("Reset complete")